# Data pre-processing and correlation analysis

In [91]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
import os

# Add the 'src' directory to the system path
src_dir = os.path.abspath(os.path.join(os.getcwd(), "..", "..", "src"))
print(f"Adding to path: {src_dir}")
print(f"Path exists: {os.path.exists(src_dir)}")
sys.path.insert(0, src_dir)

# Now you can import the module
import standardised_pipeline_utils
from sklearn.preprocessing import MinMaxScaler
from joblib import dump, load


Adding to path: c:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\correlations\src
Path exists: True


#### Datasets Used:
 
CCLE genes expressions:
Proteomics:

In [95]:
transcriptomics = pd.read_csv('https://figshare.com/ndownloader/files/25797014')
# contains expression values for around 52k genes
# non-full version includes 19k protein-coding genes
# both are log_2(x+1) transformed
transcriptomics.shape

KeyboardInterrupt: 

In [133]:
proteomics = pd.read_csv('data/protein_quant_current_normalized.csv')
# this is the normalized protein abundance data
# there is a non-normalized version
# the normalization is good
# this is what upadhya uses
proteomics.head()


,Protein_Id,Gene_Symbol,Description,Group_ID,Uniprot,Uniprot_Acc,TenPx01_Peptides,TenPx02_Peptides,TenPx03_Peptides,TenPx04_Peptides,...,NCIH2030_LUNG_TenPx18,22RV1_PROSTATE_TenPx24,A172_CENTRAL_NERVOUS_SYSTEM_TenPx24,BT20_BREAST_TenPx24,CALU6_LUNG_TenPx24,FADU_UPPER_AERODIGESTIVE_TRACT_TenPx24,KP4_PANCREAS_TenPx24,MONOMAC6_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE_TenPx24,OVCAR8_OVARY_TenPx24,THP1_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE_TenPx24
0,sp|P55011|S12A2_HUMAN,SLC12A2,S12A2_HUMAN Solute carrier family 12 member 2,0,S12A2_HUMAN,P55011,41,16,23,59,...,-1.345024,1.811205,-0.921176,-1.045727,0.845574,-0.071063,0.702306,-1.395878,-1.165032,-0.547536
1,sp|P35453|HXD13_HUMAN,HOXD13,HXD13_HUMAN Homeobox protein Hox-D13,1,HXD13_HUMAN,P35453,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,sp|O60341|KDM1A_HUMAN,KDM1A,KDM1A_HUMAN Lysine-specific histone demethylas...,2,KDM1A_HUMAN,O60341,25,35,37,40,...,0.161196,-0.337994,-0.455601,0.392393,0.258380,-0.817245,-0.790010,0.929522,-0.828736,0.735431
3,sp|O00570|SOX1_HUMAN,SOX1,SOX1_HUMAN Transcription factor SOX-1,4,SOX1_HUMAN,O00570,0,0,1,0,...,-0.040198,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,sp|P48431|SOX2_HUMAN,SOX2,SOX2_HUMAN Transcription factor SOX-2,4,SOX2_HUMAN,P48431,1,0,2,1,...,-4.627606,-1.261703,1.136851,-3.179027,1.424570,2.549687,4.424095,-1.083698,-1.640990,-2.905547


In [134]:
plaur_rows = proteomics[proteomics['Gene_Symbol'] == 'PLAUR'][['Protein_Id', 'Group_ID', 'Uniprot_Acc']]


In [135]:
plaur_rows

,Protein_Id,Group_ID,Uniprot_Acc
9700,sp|Q03405|UPAR_HUMAN,9303,Q03405
9701,sp|Q03405-2|UPAR_HUMAN,9303,Q03405-2


In [ ]:
sample_info = pd.read_csv('https://figshare.com/ndownloader/files/25494443')

### Transcriptomics: Upadhya processing and saving

In [ ]:
transcriptomics = transcriptomics.set_index('Unnamed: 0')
transcriptomics = transcriptomics.rename(index = dict(zip(sample_info['DepMap_ID'],
                                                          sample_info['CCLE_Name'])),
                                        columns = lambda x : str(x).split(' ')[0])
# changed cell line names
transcriptomics.head()

,TSPAN6,TNMD,DPM1,SCYL3,FIRRM,FGR,CFH,FUCA2,GCLC,NFYA,...,H3C3,CIMIP3,SMIM42,NPBWR1,ACTL10,PANO1,HRURF,PRRC2B,F8A2,F8A1
Unnamed: 0,,,,,,,,,,,,,,,,,,,,,
LC1SQSF_LUNG,4.956577,0.000000,7.577648,3.179411,4.765742,0.037483,1.510547,3.103039,6.697482,4.852304,...,0.093516,0.0,0.0,0.414727,0.077634,0.411901,0.0,5.134808,1.214541,4.315653
COGAR359_SOFT_TISSUE,4.954992,0.617243,7.334747,2.783576,3.736280,0.000000,0.149060,3.855960,4.377061,3.466249,...,0.672134,0.0,0.0,0.029840,0.000000,1.026871,0.0,5.951231,0.007227,4.251060
COLO794_SKIN,3.421952,0.000000,7.546069,2.615880,4.476233,0.064571,1.397491,6.833915,3.980336,3.437154,...,0.689369,0.0,0.0,0.127768,0.105594,0.540575,0.0,4.205971,0.030894,2.783448
ACH-001979,4.651643,0.000000,5.946408,2.454515,1.852111,0.000000,7.680796,6.156866,5.042574,3.258451,...,0.000000,0.0,0.0,0.012012,0.000000,0.729221,0.0,4.922916,0.150926,4.661556
ACH-002438,4.336705,0.000000,6.879387,2.262824,3.256491,0.000000,2.253926,6.895436,3.470992,4.308375,...,0.278721,0.0,0.0,0.024008,0.984538,0.549131,0.0,4.949465,0.000000,6.202939


In [ ]:
transcriptomics = transcriptomics.transpose()
assert len(transcriptomics.columns[transcriptomics.columns.duplicated()]) == 0, "columns contain duplicates"

In [ ]:
# 1. Your original analytes:
ccle_targets = [
    "EGF", "Angiopoietin‑2", "G‑CSF", "BMP‑9", "Endoglin", "Endothelin‑1",
    "Leptin", "Follistatin", "IL‑8", "HGF", "HB‑EGF", "PLGF",
    "VEGF‑C", "VEGF‑D", "FGF2", "VEGF‑A", "Angiostatin", "sAXL",
    "sc‑Kit/SCFR", "sHer2", "sHer3", "sE‑Selectin", "sHGFR/c‑Met",
    "PDGF‑AB/BB", "sIL‑6Rα", "sTie‑2", "Thrombospondin‑2",
    "sNeuropilin‑1", "sEGFR", "suPAR", "sVEGFR1", "sVEGFR3",
    "sPECAM‑1", "Osteopontin", "sVEGFR2"
]

# 2. Mapping your labels → CCLE (HGNC) gene symbols:
ccle_gene_symbol = {
    "EGF":       "EGF",

    "Angiopoietin‑2": "ANGPT2",       # encoded by ANGPT2 gene
    "G‑CSF":           "CSF3",         # gene symbol for granulocyte colony‑stimulating factor
    "BMP‑9":           "BMP9",         # bone morphogenetic protein 9
    "Endoglin":        "ENG",          # Endoglin
    "Endothelin‑1":    "EDN1",         # gene EDN1
    "Leptin":          "LEP",          # leptin
    "Follistatin":     "FST",          # follistatin
    "IL‑8":            "IL8",          # gene IL8
    "HGF":             "HGF",          # hepatocyte growth factor
    "HB‑EGF":          "HBEGF",        # heparin-binding EGF-like growth factor
    "PLGF":            "PGF",          # placenta growth factor
    "VEGF‑C":          "VEGFC",        # vascular endothelial growth factor C
    "VEGF‑D":          "VEGFD",        # vascular endothelial growth factor D
    "FGF2":            "FGF2",         # fibroblast growth factor 2
    "VEGF‑A":          "VEGFA",        # vascular endothelial growth factor A
    "Angiostatin":     "PLG",          # fragment of plasminogen (gene PLG)

    # Soluble receptors: the gene is the full receptor
    "sAXL":            "AXL",          # soluble AXL
    "sc‑Kit/SCFR":     "KIT",          # stem cell factor receptor (c‑Kit)
    "sHer2":           "ERBB2",        # soluble HER2 (extracellular domain)
    "sHer3":           "ERBB3",        # soluble HER3
    "sE‑Selectin":     "SELE",         # E‑selectin gene symbol
    "sHGFR/c‑Met":     "MET",          # hepatocyte growth factor receptor
    "PDGF‑AB/BB":      "PDGFB",        # PDGF‑BB is encoded by PDGFB (homodimer)
    "sIL‑6Rα":         "IL6R",         # IL‑6 receptor alpha chain
    "sTie‑2":          "TEK",          # Tie‑2 receptor gene
    "Thrombospondin‑2": "THBS2",       # thrombospondin 2
    "sNeuropilin‑1":   "NRP1",         # neuropilin‑1 gene
    "sEGFR":           "EGFR",         # EGFR (Her1)
    "suPAR":           "PLAUR",        # urokinase receptor (uPAR)
    "sVEGFR1":         "FLT1",         # soluble VEGFR‑1
    "sVEGFR3":         "FLT4",         # soluble VEGFR‑3
    "sPECAM‑1":        "PECAM1",       # soluble PECAM‑1 receptor molecule
    "Osteopontin":     "SPP1",         # osteopontin gene
    "sVEGFR2":         "KDR"           # soluble VEGFR‑2 (also known as KDR)
}

def to_ccle(analyte_name: str) -> str | None:
    """
    Convert your analyte name (as in ccle_targets) to the standard CCLE/HGNC gene symbol,
    or return None if not in the list.
    """
    return ccle_gene_symbol.get(analyte_name)

prot = list(ccle_gene_symbol.values())
#for el in ['CSF3', 'BMP9', 'LEP', 'IL8', 'PLG', 'SELE', 'TEK', 'FLT4', 'KDR' ]:
#   prot.remove(el)
for el in ['EGF', 'EDN1', 'PGF', 'VEGFD', 'PDGFB', 'CSF3', 'LEP', 'SELE', 'TEK',  'IL6R', 'BMP9', 'IL8']:
    prot.remove(el)


In [ ]:
# now rows are EGF, ANGPT2, CSF3, …
transcr = transcriptomics.loc[prot]

In [ ]:
transcr

Unnamed: 0,LC1SQSF_LUNG,COGAR359_SOFT_TISSUE,COLO794_SKIN,ACH-001979,ACH-002438,KE97_STOMACH,BFTC909_KIDNEY,KCIMOH1_PANCREAS,KKU213_BILIARY_TRACT,RT4_URINARY_TRACT,...,WM4235_SKIN,ACHN_KIDNEY,COLO792_SKIN,ECC2_BILIARY_TRACT,A673_BONE,ACH-002996,ACH-003142,42MGBA_CENTRAL_NERVOUS_SYSTEM,NCIH1436_LUNG,LU65_LUNG
ANGPT2,0.073534,0.035827,0.744177,0.060143,0.071608,0.102696,0.214392,0.085688,0.083152,0.169271,...,0.045699,0.077432,0.033944,0.011292,0.210447,0.010498,0.089978,0.094041,2.561434,0.130783
ENG,1.176846,0.120726,4.609377,4.716118,4.937151,1.243279,5.644505,2.880347,2.171703,0.580031,...,4.711192,0.765298,4.335276,2.657158,3.850929,3.965396,6.093928,4.803837,0.905909,0.830166
FST,1.437824,3.711605,5.755264,4.918672,2.248720,0.046100,0.114004,2.915259,4.914478,4.217409,...,2.853784,0.082911,1.301213,1.279614,0.053225,4.060186,4.197920,8.306198,0.180325,0.674280
HGF,0.137636,1.432430,0.104492,0.014859,0.000000,2.487170,0.175941,0.000000,0.000000,0.160166,...,0.648167,0.379253,0.011486,0.000000,0.343296,0.000000,2.584556,0.017087,1.692940,2.471052
HBEGF,5.224418,2.981920,3.129865,3.002600,1.141904,0.360118,3.407184,1.858685,5.654879,1.654921,...,3.887798,3.531806,2.423269,5.927195,2.390079,5.943928,4.289193,3.102940,0.740775,4.530920
VEGFC,1.173633,1.035068,2.009129,0.000000,0.000000,0.014641,4.231409,0.000000,0.398732,2.208055,...,4.979038,2.894298,0.010493,0.002770,0.143768,5.024945,5.172680,4.920932,2.162336,0.137519
FGF2,2.325145,4.659325,1.974566,0.148913,5.039976,0.361796,2.529089,3.398510,0.040860,0.178244,...,1.840395,6.523913,1.143877,0.094247,2.611475,4.063589,6.472800,1.259483,2.099080,2.764221
VEGFA,6.230987,5.786743,5.988223,5.409517,4.335569,5.707966,6.055439,5.447979,8.143694,5.842496,...,6.017138,4.724225,1.047322,5.168546,6.294080,8.049260,6.142387,7.143057,5.915048,6.092565
PLG,0.115477,0.110708,0.070844,0.000000,0.229133,0.067822,0.062958,0.000000,0.000000,0.264557,...,0.000000,0.000000,0.000000,0.235307,0.000000,0.768309,0.000000,0.153471,0.111261,0.772573
AXL,2.769653,1.214945,3.625633,0.036180,6.377946,0.100289,7.612205,6.495217,5.245092,0.199330,...,4.918162,6.837060,1.764720,6.950293,1.017851,8.346008,7.630486,6.924112,0.120783,2.851543


### Proteomics: Upadhya processing and saving

In [130]:
proteomics.set_index('Gene_Symbol', inplace=True)

In [131]:
proteomics

,Protein_Id,Description,Group_ID,Uniprot,Uniprot_Acc,TenPx01_Peptides,TenPx02_Peptides,TenPx03_Peptides,TenPx04_Peptides,TenPx06_Peptides,...,NCIH2030_LUNG_TenPx18,22RV1_PROSTATE_TenPx24,A172_CENTRAL_NERVOUS_SYSTEM_TenPx24,BT20_BREAST_TenPx24,CALU6_LUNG_TenPx24,FADU_UPPER_AERODIGESTIVE_TRACT_TenPx24,KP4_PANCREAS_TenPx24,MONOMAC6_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE_TenPx24,OVCAR8_OVARY_TenPx24,THP1_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE_TenPx24
Gene_Symbol,,,,,,,,,,,,,,,,,,,,,
SLC12A2,sp|P55011|S12A2_HUMAN,S12A2_HUMAN Solute carrier family 12 member 2,0,S12A2_HUMAN,P55011,41,16,23,59,44,...,-1.345024,1.811205,-0.921176,-1.045727,0.845574,-0.071063,0.702306,-1.395878,-1.165032,-0.547536
HOXD13,sp|P35453|HXD13_HUMAN,HXD13_HUMAN Homeobox protein Hox-D13,1,HXD13_HUMAN,P35453,0,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
KDM1A,sp|O60341|KDM1A_HUMAN,KDM1A_HUMAN Lysine-specific histone demethylas...,2,KDM1A_HUMAN,O60341,25,35,37,40,33,...,0.161196,-0.337994,-0.455601,0.392393,0.258380,-0.817245,-0.790010,0.929522,-0.828736,0.735431
SOX1,sp|O00570|SOX1_HUMAN,SOX1_HUMAN Transcription factor SOX-1,4,SOX1_HUMAN,O00570,0,0,1,0,0,...,-0.040198,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
SOX2,sp|P48431|SOX2_HUMAN,SOX2_HUMAN Transcription factor SOX-2,4,SOX2_HUMAN,P48431,1,0,2,1,2,...,-4.627606,-1.261703,1.136851,-3.179027,1.424570,2.549687,4.424095,-1.083698,-1.640990,-2.905547
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
CCL26,sp|Q9Y258|CCL26_HUMAN,CCL26_HUMAN C-C motif chemokine 26,23903,CCL26_HUMAN,Q9Y258,0,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ALOX5AP,sp|P20292|AL5AP_HUMAN,AL5AP_HUMAN Arachidonate 5-lipoxygenase-activa...,24093,AL5AP_HUMAN,P20292,0,0,0,0,0,...,NaN,-0.690046,-1.600231,-1.110583,-1.047633,0.023850,-1.393552,4.556818,-0.531699,0.640295
CYSTM1,sp|Q9H1C7|CYTM1_HUMAN,CYTM1_HUMAN Cysteine-rich and transmembrane do...,24337,CYTM1_HUMAN,Q9H1C7,3,1,1,1,0,...,-0.393738,0.604400,1.643414,-1.481004,0.408420,-0.177391,0.445977,-2.497558,0.964551,-2.004067


In [129]:
proteomics.loc['PLAUR']

KeyError: 'PLAUR'

In [122]:
proteomics = proteomics.loc[:, proteomics.columns.str.contains('_TenPx')]

In [123]:
proteomics

,MDAMB468_BREAST_TenPx01,SH4_SKIN_TenPx01,AU565_BREAST_TenPx01,KMRC1_KIDNEY_TenPx01,CAL51_BREAST_TenPx01,RPMI7951_SKIN_TenPx01,RERFLCMS_LUNG_TenPx01,IGR37_SKIN_TenPx01,VMRCRCW_KIDNEY_TenPx01,HEP3B217_LIVER_TenPx02,...,NCIH2030_LUNG_TenPx18,22RV1_PROSTATE_TenPx24,A172_CENTRAL_NERVOUS_SYSTEM_TenPx24,BT20_BREAST_TenPx24,CALU6_LUNG_TenPx24,FADU_UPPER_AERODIGESTIVE_TRACT_TenPx24,KP4_PANCREAS_TenPx24,MONOMAC6_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE_TenPx24,OVCAR8_OVARY_TenPx24,THP1_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE_TenPx24
Gene_Symbol,,,,,,,,,,,,,,,,,,,,,
SLC12A2,2.111348,0.070468,-0.463928,-0.883645,0.788565,-0.912352,0.729167,-0.658213,-1.150653,0.008824,...,-1.345024,1.811205,-0.921176,-1.045727,0.845574,-0.071063,0.702306,-1.395878,-1.165032,-0.547536
HOXD13,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
KDM1A,0.379683,-0.283538,0.191211,-0.595606,0.391243,-0.381329,0.410168,0.066076,-0.324100,0.316569,...,0.161196,-0.337994,-0.455601,0.392393,0.258380,-0.817245,-0.790010,0.929522,-0.828736,0.735431
SOX1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,-0.040198,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
SOX2,-0.246367,-0.572753,-0.341305,-0.335211,-0.124474,0.043893,1.882953,-0.616307,-0.533335,NaN,...,-4.627606,-1.261703,1.136851,-3.179027,1.424570,2.549687,4.424095,-1.083698,-1.640990,-2.905547
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
CCL26,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ALOX5AP,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,-0.690046,-1.600231,-1.110583,-1.047633,0.023850,-1.393552,4.556818,-0.531699,0.640295
CYSTM1,-0.313512,0.105086,-0.586815,-0.132940,0.452423,-0.546389,0.298349,0.461098,0.309062,0.017985,...,-0.393738,0.604400,1.643414,-1.481004,0.408420,-0.177391,0.445977,-2.497558,0.964551,-2.004067


In [124]:
# Checking for cell lines repeated in >1 Ten-plexes
proteomics.filter(regex='SW948_LARGE_INTESTINE|CAL120_BREAST|HCT15_LARGE_INTESTINE').columns

Index(['CAL120_BREAST_TenPx02', 'SW948_LARGE_INTESTINE_TenPx11',
       'SW948_LARGE_INTESTINE_TenPx20', 'CAL120_BREAST_TenPx28',
       'HCT15_LARGE_INTESTINE_TenPx30', 'HCT15_LARGE_INTESTINE_TenPx18'],
      dtype='object')

In [125]:
# Eliminating the cell lines that do not correlate well with transcriptomics data as mentioned in the paper 
proteomics.drop(columns=['SW948_LARGE_INTESTINE_TenPx11', 'CAL120_BREAST_TenPx02', 'HCT15_LARGE_INTESTINE_TenPx30'], 
                    inplace=True)
proteomics = proteomics.rename(columns = lambda x : str(x).split('_TenPx')[0])

C:\Users\hp\AppData\Local\Temp\ipykernel_32356\283973742.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  proteomics.drop(columns=['SW948_LARGE_INTESTINE_TenPx11', 'CAL120_BREAST_TenPx02', 'HCT15_LARGE_INTESTINE_TenPx30'],


In [109]:
assert len(proteomics.columns[proteomics.columns.duplicated()]) == 0, "columns contain duplicates"

In [ ]:
#proteomics_processed = standardised_pipeline_utils.process(proteomics)
# drop rows (proteins) with more than 20% NaNs (removed 12755-7372=5383 proteins)
# rows with same index are collapsed together into one row with avg values
# drop rows that are indexed by datetime object
# drop rows containing ":"
# proteomics_processed.head()

In [126]:
proteomics.loc['PLAUR']

,MDAMB468_BREAST,SH4_SKIN,AU565_BREAST,KMRC1_KIDNEY,CAL51_BREAST,RPMI7951_SKIN,RERFLCMS_LUNG,IGR37_SKIN,VMRCRCW_KIDNEY,HEP3B217_LIVER,...,NCIH2030_LUNG,22RV1_PROSTATE,A172_CENTRAL_NERVOUS_SYSTEM,BT20_BREAST,CALU6_LUNG,FADU_UPPER_AERODIGESTIVE_TRACT,KP4_PANCREAS,MONOMAC6_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,OVCAR8_OVARY,THP1_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE
Gene_Symbol,,,,,,,,,,,,,,,,,,,,,
PLAUR,-0.351529,-0.581176,-0.633794,0.961357,-0.891422,1.680244,0.588364,-0.712463,0.20849,-1.278113,...,1.653384,-1.321572,1.788581,-0.084902,0.828481,-0.670258,0.689986,0.540209,-0.124760,-0.134630
PLAUR,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.458100,...,1.239744,0.273193,-0.162342,0.160560,0.691599,0.865625,0.500788,0.779662,0.063648,-0.869416


In [ ]:
# now rows are EGF, ANGPT2, CSF3, …
proteo = proteomics.loc[prot]

Follows cell to save datasets in a 'data 'folder

In [ ]:
proteo.to_csv('data/proteo.csv', index=False)
transcr.to_csv('data/transcr.csv', index=False)

In [ ]:
proteo

,MDAMB468_BREAST,SH4_SKIN,AU565_BREAST,KMRC1_KIDNEY,CAL51_BREAST,RPMI7951_SKIN,RERFLCMS_LUNG,IGR37_SKIN,VMRCRCW_KIDNEY,HEP3B217_LIVER,...,NCIH2030_LUNG,22RV1_PROSTATE,A172_CENTRAL_NERVOUS_SYSTEM,BT20_BREAST,CALU6_LUNG,FADU_UPPER_AERODIGESTIVE_TRACT,KP4_PANCREAS,MONOMAC6_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,OVCAR8_OVARY,THP1_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE
Gene_Symbol,,,,,,,,,,,,,,,,,,,,,
ANGPT2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ENG,-0.375677,0.202232,0.074901,0.548786,-0.940937,1.119791,-0.339454,0.267067,-0.426567,-1.330305,...,-0.485977,-1.772145,2.309206,0.330536,1.399800,-0.044471,-1.523498,0.123122,-0.365863,0.201467
FST,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
HGF,-0.232176,0.014172,-0.762811,-0.927583,0.734210,-0.016255,0.965357,0.276607,-1.158560,NaN,...,-0.107388,1.861547,-1.104364,-0.992204,-0.883751,-0.716809,4.238678,0.115918,-0.958941,-0.328587
HBEGF,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
VEGFC,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,0.105960,0.231409,-0.998181,0.016262,0.067198,1.432967,-0.048055,0.088998,-0.336371
FGF2,-1.315002,-0.295736,-1.231739,1.742687,0.305717,0.167197,0.163756,-1.612004,1.854341,-1.175239,...,-0.067329,-0.323582,4.538873,-0.762451,-3.154367,-0.617847,-0.189354,-2.227347,2.530288,-1.625983
VEGFA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,-0.299738,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
PLG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.364999,-0.833414,-0.987061,-2.048411,1.323548,0.447093,0.955481,2.205684,-0.889007,0.096386


In [ ]:
transcr

Unnamed: 0,LC1SQSF_LUNG,COGAR359_SOFT_TISSUE,COLO794_SKIN,ACH-001979,ACH-002438,KE97_STOMACH,BFTC909_KIDNEY,KCIMOH1_PANCREAS,KKU213_BILIARY_TRACT,RT4_URINARY_TRACT,...,WM4235_SKIN,ACHN_KIDNEY,COLO792_SKIN,ECC2_BILIARY_TRACT,A673_BONE,ACH-002996,ACH-003142,42MGBA_CENTRAL_NERVOUS_SYSTEM,NCIH1436_LUNG,LU65_LUNG
ANGPT2,0.073534,0.035827,0.744177,0.060143,0.071608,0.102696,0.214392,0.085688,0.083152,0.169271,...,0.045699,0.077432,0.033944,0.011292,0.210447,0.010498,0.089978,0.094041,2.561434,0.130783
ENG,1.176846,0.120726,4.609377,4.716118,4.937151,1.243279,5.644505,2.880347,2.171703,0.580031,...,4.711192,0.765298,4.335276,2.657158,3.850929,3.965396,6.093928,4.803837,0.905909,0.830166
FST,1.437824,3.711605,5.755264,4.918672,2.248720,0.046100,0.114004,2.915259,4.914478,4.217409,...,2.853784,0.082911,1.301213,1.279614,0.053225,4.060186,4.197920,8.306198,0.180325,0.674280
HGF,0.137636,1.432430,0.104492,0.014859,0.000000,2.487170,0.175941,0.000000,0.000000,0.160166,...,0.648167,0.379253,0.011486,0.000000,0.343296,0.000000,2.584556,0.017087,1.692940,2.471052
HBEGF,5.224418,2.981920,3.129865,3.002600,1.141904,0.360118,3.407184,1.858685,5.654879,1.654921,...,3.887798,3.531806,2.423269,5.927195,2.390079,5.943928,4.289193,3.102940,0.740775,4.530920
VEGFC,1.173633,1.035068,2.009129,0.000000,0.000000,0.014641,4.231409,0.000000,0.398732,2.208055,...,4.979038,2.894298,0.010493,0.002770,0.143768,5.024945,5.172680,4.920932,2.162336,0.137519
FGF2,2.325145,4.659325,1.974566,0.148913,5.039976,0.361796,2.529089,3.398510,0.040860,0.178244,...,1.840395,6.523913,1.143877,0.094247,2.611475,4.063589,6.472800,1.259483,2.099080,2.764221
VEGFA,6.230987,5.786743,5.988223,5.409517,4.335569,5.707966,6.055439,5.447979,8.143694,5.842496,...,6.017138,4.724225,1.047322,5.168546,6.294080,8.049260,6.142387,7.143057,5.915048,6.092565
PLG,0.115477,0.110708,0.070844,0.000000,0.229133,0.067822,0.062958,0.000000,0.000000,0.264557,...,0.000000,0.000000,0.000000,0.235307,0.000000,0.768309,0.000000,0.153471,0.111261,0.772573
AXL,2.769653,1.214945,3.625633,0.036180,6.377946,0.100289,7.612205,6.495217,5.245092,0.199330,...,4.918162,6.837060,1.764720,6.950293,1.017851,8.346008,7.630486,6.924112,0.120783,2.851543


### Matching cell lines

In [ ]:
matched_cell_lines = [line for line in proteo.transpose().index if line in transcr.transpose().index]
len(matched_cell_lines)

370

In [ ]:
transcriptomics_matched = transcr.transpose().reindex(matched_cell_lines)
proteomics_matched = proteo.transpose().reindex(matched_cell_lines)
# only keeping the 369 cell lines present both in transcriptomics and proteomics data

In [ ]:
transcriptomics_matched

,ANGPT2,ENG,FST,HGF,HBEGF,VEGFC,FGF2,VEGFA,PLG,AXL,...,MET,THBS2,NRP1,EGFR,PLAUR,FLT1,FLT4,PECAM1,SPP1,KDR
Unnamed: 0,,,,,,,,,,,,,,,,,,,,,
MDAMB468_BREAST,0.030808,3.598377,1.278003,0.157800,2.503850,0.126516,2.259878,5.850812,0.000000,1.118167,...,5.206663,0.264757,3.693699,9.119836,5.580176,0.278992,0.009817,0.456627,1.994073,0.832820
SH4_SKIN,0.045401,4.395392,2.501831,0.083998,3.814889,0.997625,1.604166,7.895763,0.000000,0.461933,...,5.716138,7.209531,1.324465,0.154632,5.319210,0.115413,0.715856,0.035548,7.907624,0.114440
AU565_BREAST,0.029386,5.069075,0.151250,0.039029,0.789814,1.185968,0.157352,5.738652,0.013967,0.289371,...,5.252586,0.125896,3.497395,4.938719,4.003701,0.032095,0.019459,0.086551,1.507749,0.070507
KMRC1_KIDNEY,0.026902,4.282069,0.169114,0.106565,3.085559,5.528778,3.595074,7.318715,0.050426,2.505766,...,6.016926,0.053081,5.165120,4.399479,7.681140,0.660725,0.041686,0.016737,9.644700,0.030794
CAL51_BREAST,0.101076,0.290932,0.329568,0.316533,1.274709,0.590316,3.394398,6.730205,0.010495,2.507937,...,5.478101,0.127546,6.724527,3.711583,2.240436,0.373992,3.439146,0.603774,9.375417,0.021121
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
FADU_UPPER_AERODIGESTIVE_TRACT,0.119128,3.408363,3.054651,0.141343,1.776748,1.019211,1.686631,5.318939,0.010425,3.180361,...,5.570226,0.040573,2.351357,6.614006,2.301848,0.208970,0.283607,0.000000,0.781713,0.000000
KP4_PANCREAS,0.069379,1.885711,0.262953,9.313878,2.714386,6.537444,2.400060,5.862875,0.059423,4.076140,...,7.240791,0.146430,3.830702,3.382858,4.723897,0.010009,0.263777,0.024964,9.330343,0.023187
MONOMAC6_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,0.078432,4.463812,0.195380,1.398235,4.235038,0.335981,1.133787,8.226489,0.000000,0.156992,...,0.239574,0.143830,0.243827,0.122564,5.237223,2.992973,0.716143,5.564207,3.112003,0.000000


In [ ]:
dump(transcriptomics_matched, "data/transcr_matched.joblib")
dump(proteomics_matched, "data/proteo_matched.joblib")

['data/proteo_matched.joblib']

In [ ]:
transcriptomics_matched.shape, proteomics_matched.shape

((370, 23), (370, 24))

### Correlations

In [97]:
transcriptomics = load('data/transcr_matched.joblib')
proteomics = load('data/proteo_matched.joblib')

#### Pearson 

In [98]:
transcriptomics

,ANGPT2,ENG,FST,HGF,HBEGF,VEGFC,FGF2,VEGFA,PLG,AXL,...,MET,THBS2,NRP1,EGFR,PLAUR,FLT1,FLT4,PECAM1,SPP1,KDR
Unnamed: 0,,,,,,,,,,,,,,,,,,,,,
MDAMB468_BREAST,0.030808,3.598377,1.278003,0.157800,2.503850,0.126516,2.259878,5.850812,0.000000,1.118167,...,5.206663,0.264757,3.693699,9.119836,5.580176,0.278992,0.009817,0.456627,1.994073,0.832820
SH4_SKIN,0.045401,4.395392,2.501831,0.083998,3.814889,0.997625,1.604166,7.895763,0.000000,0.461933,...,5.716138,7.209531,1.324465,0.154632,5.319210,0.115413,0.715856,0.035548,7.907624,0.114440
AU565_BREAST,0.029386,5.069075,0.151250,0.039029,0.789814,1.185968,0.157352,5.738652,0.013967,0.289371,...,5.252586,0.125896,3.497395,4.938719,4.003701,0.032095,0.019459,0.086551,1.507749,0.070507
KMRC1_KIDNEY,0.026902,4.282069,0.169114,0.106565,3.085559,5.528778,3.595074,7.318715,0.050426,2.505766,...,6.016926,0.053081,5.165120,4.399479,7.681140,0.660725,0.041686,0.016737,9.644700,0.030794
CAL51_BREAST,0.101076,0.290932,0.329568,0.316533,1.274709,0.590316,3.394398,6.730205,0.010495,2.507937,...,5.478101,0.127546,6.724527,3.711583,2.240436,0.373992,3.439146,0.603774,9.375417,0.021121
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
FADU_UPPER_AERODIGESTIVE_TRACT,0.119128,3.408363,3.054651,0.141343,1.776748,1.019211,1.686631,5.318939,0.010425,3.180361,...,5.570226,0.040573,2.351357,6.614006,2.301848,0.208970,0.283607,0.000000,0.781713,0.000000
KP4_PANCREAS,0.069379,1.885711,0.262953,9.313878,2.714386,6.537444,2.400060,5.862875,0.059423,4.076140,...,7.240791,0.146430,3.830702,3.382858,4.723897,0.010009,0.263777,0.024964,9.330343,0.023187
MONOMAC6_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,0.078432,4.463812,0.195380,1.398235,4.235038,0.335981,1.133787,8.226489,0.000000,0.156992,...,0.239574,0.143830,0.243827,0.122564,5.237223,2.992973,0.716143,5.564207,3.112003,0.000000


In [ ]:
pearson_correlations = transcriptomics.corrwith(proteomics,
                                                      axis=0,
                                                      method='pearson')

In [ ]:
pearson_correlations.describe()

count    24.000000
mean      0.640809
std       0.231519
min      -0.033667
25%       0.596816
50%       0.678678
75%       0.803999
max       0.861169
dtype: float64

#### Spearman

In [ ]:
spearman_correlations = transcriptomics.corrwith(proteomics,
                                                      axis=0,
                                                      method='spearman')

In [100]:
proteomics

Gene_Symbol,ANGPT2,ENG,FST,HGF,HBEGF,VEGFC,FGF2,VEGFA,PLG,AXL,...,THBS2,NRP1,EGFR,PLAUR,PLAUR,FLT1,FLT4,PECAM1,SPP1,KDR
MDAMB468_BREAST,NaN,-0.375677,NaN,-0.232176,NaN,NaN,-1.315002,NaN,NaN,-1.019767,...,-0.654645,-0.892394,2.649190,-0.351529,NaN,NaN,0.068819,NaN,-1.147969,NaN
SH4_SKIN,NaN,0.202232,NaN,0.014172,NaN,NaN,-0.295736,NaN,NaN,-0.556712,...,1.909693,-1.203970,-1.120259,-0.581176,NaN,NaN,0.448495,NaN,0.088925,NaN
AU565_BREAST,NaN,0.074901,NaN,-0.762811,NaN,NaN,-1.231739,NaN,NaN,-1.346715,...,-0.491267,-1.205142,-0.508845,-0.633794,NaN,NaN,0.093043,NaN,-1.077979,NaN
KMRC1_KIDNEY,NaN,0.548786,NaN,-0.927583,NaN,NaN,1.742687,NaN,NaN,-0.512002,...,0.038915,0.368088,0.908958,0.961357,NaN,NaN,-0.104221,NaN,0.191855,NaN
CAL51_BREAST,NaN,-0.940937,NaN,0.734210,NaN,NaN,0.305717,NaN,NaN,-0.494707,...,-0.251203,1.043825,-0.455998,-0.891422,NaN,NaN,0.003605,NaN,-0.038532,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
FADU_UPPER_AERODIGESTIVE_TRACT,NaN,-0.044471,NaN,-0.716809,NaN,0.067198,-0.617847,NaN,0.447093,-0.096367,...,-0.041241,-1.472947,0.549699,-0.670258,0.865625,NaN,-1.021505,-0.837682,-1.398101,NaN
KP4_PANCREAS,NaN,-1.523498,NaN,4.238678,NaN,1.432967,-0.189354,NaN,0.955481,-0.872547,...,-0.397354,-0.629629,-0.734306,0.689986,0.500788,NaN,-1.236058,-1.451669,5.244891,NaN
MONOMAC6_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,NaN,0.123122,NaN,0.115918,NaN,-0.048055,-2.227347,NaN,2.205684,-1.203508,...,-1.241507,-1.374891,-2.489687,0.540209,0.779662,NaN,1.168418,3.378580,1.274474,NaN
OVCAR8_OVARY,NaN,-0.365863,NaN,-0.958941,NaN,0.088998,2.530288,NaN,-0.889007,2.088693,...,-0.268474,-1.195804,0.653639,-0.124760,0.063648,NaN,-0.237692,-1.252702,-0.650290,NaN


In [ ]:
spearman_correlations.describe()

count    24.000000
mean      0.571230
std       0.277023
min      -0.094267
25%       0.357106
50%       0.691249
75%       0.801213
max       0.865220
dtype: float64

#### Kendall

In [ ]:
kendall_correlations = transcriptomics.corrwith(proteomics,
                                                      axis=0,
                                                      method='kendall')

In [ ]:
kendall_correlations.describe()

count    24.000000
mean      0.418352
std       0.211740
min      -0.062267
25%       0.238994
50%       0.498003
75%       0.600129
max       0.667502
dtype: float64

In [ ]:
correlations = pd.DataFrame({'pearson': pearson_correlations,
                             'spearman': spearman_correlations,
                             'kendall': kendall_correlations})

In [ ]:
correlations

,pearson,spearman,kendall
ANGPT2,0.657328,0.785346,0.594771
AXL,0.822503,0.824989,0.612605
EGFR,0.861169,0.865220,0.667502
ENG,0.746277,0.837814,0.626777
ERBB2,0.848898,0.786410,0.599385
ERBB3,0.774351,0.800579,0.580551
FGF2,0.764983,0.808381,0.602363
FLT1,0.648572,0.284079,0.202878
FLT4,-0.033667,-0.094267,-0.062267
FST,0.803452,0.735294,0.535075


In [ ]:
ccle_gene_symbol

{'EGF': 'EGF',
 'Angiopoietin‑2': 'ANGPT2',
 'G‑CSF': 'CSF3',
 'BMP‑9': 'BMP9',
 'Endoglin': 'ENG',
 'Endothelin‑1': 'EDN1',
 'Leptin': 'LEP',
 'Follistatin': 'FST',
 'IL‑8': 'IL8',
 'HGF': 'HGF',
 'HB‑EGF': 'HBEGF',
 'PLGF': 'PGF',
 'VEGF‑C': 'VEGFC',
 'VEGF‑D': 'VEGFD',
 'FGF2': 'FGF2',
 'VEGF‑A': 'VEGFA',
 'Angiostatin': 'PLG',
 'sAXL': 'AXL',
 'sc‑Kit/SCFR': 'KIT',
 'sHer2': 'ERBB2',
 'sHer3': 'ERBB3',
 'sE‑Selectin': 'SELE',
 'sHGFR/c‑Met': 'MET',
 'PDGF‑AB/BB': 'PDGFB',
 'sIL‑6Rα': 'IL6R',
 'sTie‑2': 'TEK',
 'Thrombospondin‑2': 'THBS2',
 'sNeuropilin‑1': 'NRP1',
 'sEGFR': 'EGFR',
 'suPAR': 'PLAUR',
 'sVEGFR1': 'FLT1',
 'sVEGFR3': 'FLT4',
 'sPECAM‑1': 'PECAM1',
 'Osteopontin': 'SPP1',
 'sVEGFR2': 'KDR'}

In [ ]:
excluded = ['EGF', 'EDN1', 'PGF', 'VEGFD', 'PDGFB', 'CSF3', 'LEP', 'SELE', 'TEK', 'IL6R', 'BMP9', 'IL8']

# Filter the gene symbols to include only the ones not in excluded
filtered_prot = [v for k, v in ccle_gene_symbol.items() if v not in excluded]

# Create reverse mapping from HGNC symbol to analyte name
hgnc_to_analyte = {v: k for k, v in ccle_gene_symbol.items() if v in filtered_prot}

correlations.index.name = 'Protein'
correlations = correlations.copy()
correlations['HGNC symbols'] = correlations.index.map(hgnc_to_analyte)


In [ ]:
correlations

,pearson,spearman,kendall,HGNC symbols
Protein,,,,
ANGPT2,0.657328,0.785346,0.594771,Angiopoietin‑2
AXL,0.822503,0.824989,0.612605,sAXL
EGFR,0.861169,0.865220,0.667502,sEGFR
ENG,0.746277,0.837814,0.626777,Endoglin
ERBB2,0.848898,0.786410,0.599385,sHer2
ERBB3,0.774351,0.800579,0.580551,sHer3
FGF2,0.764983,0.808381,0.602363,FGF2
FLT1,0.648572,0.284079,0.202878,sVEGFR1
FLT4,-0.033667,-0.094267,-0.062267,sVEGFR3


In [94]:
correlations.to_csv('data/correlations_hypoxia.csv', index=True)

In [ ]:
dump(correlations, 'data/correlations_hypoxia.joblib')

['data/correlations_hypoxia.joblib']